# 00 — Completed research benchmark

**Question:** which open vision-language model configuration gives the strongest answer quality for its compute cost, and how reliably can text retrieval find the evidence in multilingual PDFs?

This release contains a 4B/9B/27B oracle-reader comparison, full-document BM25 retrieval, a 1,582-configuration lexical feature search, and a pinned visual-retrieval challenger. The deliverable is the measured results, reproducible Python implementation, and executed analysis. Kaggle submission and deployment are outside this release.

**For employers:** read this overview, then [03 — Model quality and cost](03_model_scaling_and_cost.ipynb) and [04 — Evidence retrieval](04_evidence_retrieval.ipynb). Notebooks 01 and 02 provide experiment-design and cloud-execution detail. Saved outputs are included; no account, installation, or rerun is needed.

## 1. Inspect the measured results

These tables are generated from verified public reports. The reader score uses supplied correct evidence pages. Retrieval measures page selection separately; it is not an end-to-end answer score.

In [1]:
import json
from pathlib import Path

from IPython.display import HTML, display

from lava.evaluation.reporting import load_report
from lava.evaluation.walkthrough import (
    TABLE_STYLE,
    comparison_tables,
    render_table,
    training_rates,
)
from lava.notebook_support import find_repo_root
from lava.readers.runtime_logging import RuntimeEventLogger
from lava.retrieval.pipeline import load_public_report

ROOT = find_repo_root(Path.cwd())
logger = RuntimeEventLogger("notebook.protocol")
with logger.stage("01_verify_benchmark_results", heartbeat_seconds=15):
    report = load_report(ROOT)
    retrieval = load_public_report(ROOT)
    pricing = json.loads((ROOT / "reports/aws/training_prices.json").read_text())
    tables = comparison_tables(report, training_rates(pricing))
    assert len(report["current_models"]) == 3
    assert all(row["Stage"] == "Full pilot scored" for row in tables["coverage"])
    assert report["expected_questions"] == retrieval["question_count"] == 16
    assert retrieval["reader_evaluated"] is False
    assert retrieval["local_lava_overall"] is None
    quality_columns = ("Reader", "Semantic VQA", "Evidence F1", "Local LAVA overall")
    display(
        HTML(
            TABLE_STYLE
            + render_table(
                [{key: row[key] for key in quality_columns} for row in tables["quality"]],
                percent_columns=quality_columns[1:],
                caption="Completed reader benchmark · correct evidence supplied",
            )
        )
    )
    retrieval_rows = []
    for method, label in (("page_order", "Page-order control"), ("bm25", "Multilingual BM25")):
        values = retrieval["methods"][method]["question_average"]["5"]
        retrieval_rows.append(
            {
                "Method": label,
                "Pages": 5,
                "Evidence recall": values["recall_at_k"],
                "All evidence found": values["all_evidence_at_k"],
            }
        )
    display(
        HTML(
            render_table(
                retrieval_rows,
                percent_columns=("Evidence recall", "All evidence found"),
                caption="Completed retrieval benchmark · five-page diagnostic",
            )
        )
    )

{"component": "notebook.protocol", "elapsed_seconds": 0.0, "event": "01_verify_benchmark_results.started", "level": "INFO", "stage_elapsed_seconds": 0.0, "timestamp_utc": "2026-09-08T00:26:30.575+00:00"}


Reader,Semantic VQA,Evidence F1,Local LAVA overall
Qwen3.5 · 4B,50.62%,97.02%,73.82%
Qwen3.5 · 9B,80.15%,93.90%,87.02%
Qwen3.8 · 27B NF4,70.98%,89.73%,80.36%


Method,Pages,Evidence recall,All evidence found
Page-order control,5,45.31%,37.50%
Multilingual BM25,5,95.31%,87.50%


{"component": "notebook.protocol", "elapsed_seconds": 0.388, "event": "01_verify_benchmark_results.completed", "level": "INFO", "stage_elapsed_seconds": 0.388, "timestamp_utc": "2026-09-08T00:26:30.963+00:00"}


## 2. Interpret the findings

**9B achieved the highest local LAVA score among the measured configurations.** The 27B run did not improve this pilot and produced one invalid response, which remains a failure in the score. Hardware, model generation, and precision differ; this experiment does not isolate parameter count.

**BM25 found more relevant evidence than page order on this pilot.** At five pages it found all evidence for 14 of 16 questions. Equal-document results and missed cases in Notebook 04 show why high average recall is not enough. The full page-budget curve is reported; five pages is a diagnostic, not a validated production setting.

The [published LAVA metric](https://lava-workshop.github.io/#evaluation) averages semantic answer credit and evidence-page F1 per question. Our pinned Gemma-3 1B judge uses a validated local prompt. The organizer's exact prompt/runtime is unpublished; local scores are not claimed to be identical to leaderboard scores.

Five PDFs support descriptive findings, not held-out performance, language-wide accuracy, or broad model superiority. No end-to-end answer score is claimed because retrieval and reading were evaluated separately.

## 3. Verify the data and evaluation unit

The audit covers 208 files and all 205 PDFs. Every supplied training label is included: 16 questions from five PDFs. Hidden test questions are catalogued for provenance; their labels are unavailable and no test quality score is reported.

**Broad feature research did not justify replacing the simple baseline.** After 349 duplicate-ranking rejections, document-isolated selection retained BM25. A visual-only challenger underperformed; a conservative hybrid recovered one more complete question (15/16) on the development pilot. Notebook 04 reports the negative findings and the hybrid's lack of independent validation.

In [2]:
with logger.stage("02_verify_data", heartbeat_seconds=15):
    manifest = json.loads((ROOT / "reports/raw_data_manifest_summary.json").read_text())
    audit = json.loads((ROOT / "reports/data_audit/data_audit_summary_full.json").read_text())
    assert manifest["complete"] and manifest["verified_file_count"] == 208
    assert audit["audit_mode"] == "full" and audit["audited_pdf_count_for_mode"] == 205
    rows = []
    for profile in audit["csv_profiles"]:
        if profile["selected_columns"]["question"] is None:
            continue
        split = "Training" if profile["selected_columns"]["answer"] else "Test"
        rows.append(
            {
                "Split": split,
                "Questions": profile["row_count"],
                "PDFs": profile["referenced_document_count"],
                "Japanese": profile["language_counts"]["ja"],
                "Vietnamese": profile["language_counts"]["vi"],
            }
        )
    display(HTML(TABLE_STYLE + render_table(rows, caption="Data audit complete")))
    display(
        HTML(
            render_table(
                [
                    {"Check": "Verified raw files", "Result": manifest["verified_file_count"]},
                    {"Check": "PDFs audited", "Result": audit["audited_pdf_count_for_mode"]},
                    {
                        "Check": "Exact PDF duplicates across splits",
                        "Result": audit["cross_split_exact_duplicate_pdf_group_count"],
                    },
                ],
                caption="Checks completed before modeling",
            )
        )
    )

{"component": "notebook.protocol", "elapsed_seconds": 0.397, "event": "02_verify_data.started", "level": "INFO", "stage_elapsed_seconds": 0.0, "timestamp_utc": "2026-09-08T00:26:30.971+00:00"}


Split,Questions,PDFs,Japanese,Vietnamese
Test,624,200,587,37
Training,16,5,15,1


Check,Result
Verified raw files,208
PDFs audited,205
Exact PDF duplicates across splits,0


{"component": "notebook.protocol", "elapsed_seconds": 0.4, "event": "02_verify_data.completed", "level": "INFO", "stage_elapsed_seconds": 0.003, "timestamp_utc": "2026-09-08T00:26:30.974+00:00"}


## 4. Review the completed deliverable

The completed component-level research benchmark includes data verification, frozen reader comparisons, local semantic and evidence metrics, full-document retrieval diagnostics, resource analysis, executed notebooks, and tested persistence/recovery.

An integrated reader/retriever experiment, deployment, and Kaggle submission are optional extensions outside this release. No additional run is required to review this work.

At closeout, the pending optional integrated GPU attempt was stopped. Its answer score remains unmeasured. This boundary is final for the portfolio release; no additional experiment is required.

In [3]:
with logger.stage("03_verify_release_scope", heartbeat_seconds=15):
    display(
        HTML(
            render_table(
                [
                    {
                        "Deliverable": "Data verification",
                        "Evidence": "208 files and 205 PDFs audited",
                    },
                    {
                        "Deliverable": "Reader comparison",
                        "Evidence": "Three full 16-question pilots scored",
                    },
                    {"Deliverable": "Retrieval evaluation", "Evidence": retrieval["status"]},
                    {
                        "Deliverable": "Model and systems analysis",
                        "Evidence": "Scores, slices, uncertainty, runtime, memory, cost",
                    },
                    {
                        "Deliverable": "Reproducible presentation",
                        "Evidence": "Six executed canonical notebooks with verified manifests",
                    },
                    {
                        "Deliverable": "Release scope",
                        "Evidence": "Completed research benchmark; submission optional",
                    },
                ],
                caption="Portfolio deliverables and their evidence",
            )
        )
    )
logger.emit(
    "protocol.walkthrough.completed",
    raw_files=manifest["verified_file_count"],
    complete_reader_pilots=len(report["current_models"]),
    retrieval_evaluated=retrieval["status"] == "Full-document retrieval evaluated",
    release_scope="component_research_benchmark",
    submission_required=False,
)

{"component": "notebook.protocol", "elapsed_seconds": 0.406, "event": "03_verify_release_scope.started", "level": "INFO", "stage_elapsed_seconds": 0.0, "timestamp_utc": "2026-09-08T00:26:30.981+00:00"}


Deliverable,Evidence
Data verification,208 files and 205 PDFs audited
Reader comparison,Three full 16-question pilots scored
Retrieval evaluation,Full-document retrieval evaluated
Model and systems analysis,"Scores, slices, uncertainty, runtime, memory, cost"
Reproducible presentation,Five executed canonical notebooks with verified manifests
Release scope,Completed research benchmark; submission optional


{"component": "notebook.protocol", "elapsed_seconds": 0.408, "event": "03_verify_release_scope.completed", "level": "INFO", "stage_elapsed_seconds": 0.002, "timestamp_utc": "2026-09-08T00:26:30.982+00:00"}


{"complete_reader_pilots": 3, "component": "notebook.protocol", "elapsed_seconds": 0.409, "event": "protocol.walkthrough.completed", "level": "INFO", "raw_files": 208, "release_scope": "component_research_benchmark", "retrieval_evaluated": true, "submission_required": false, "timestamp_utc": "2026-09-08T00:26:30.983+00:00"}


## 5. Continue only as far as you need

- [03 — Model quality and cost](03_model_scaling_and_cost.ipynb): answer quality, citations, document effects, and resource tradeoffs.
- [04 — Evidence retrieval](04_evidence_retrieval.ipynb): retrieval curves, failures, and the measured resume run.
- [01 — Experiment design](01_oracle_reader_benchmark_design.ipynb) and [02 — Cloud execution](02_verified_gpu_execution.ipynb): technical method.

All six notebooks live in `lava-aws-multilingual-docvqa/notebooks/`. Reusable implementation and tests remain in the repository. Private model, retrieval, and judge checkpoints persist in S3. GitHub preserves public code, metrics, and notebook outputs.

For optional reproduction, `make notebooks` verifies/reuses unchanged outputs and refreshes changed inputs; `make quality` runs the quality gate. Logs include UTC timestamps, stage and total elapsed time, progress, and heartbeats. Independent SageMaker jobs can outlive a browser session; Studio CPU processes stop with the app and resume from saved checkpoints.

[Project overview](../README.md) · [Evaluation and recovery](../docs/evaluation.md)